# Part 11 · Notebook 02 — News sentiment at scale: batches, caching, timestamps and event studies

**Sessions:** S2 (NLP at scale: LLM sentiment & event studies) · [Lesson plan](../../docs/lessons/PART_11_AI_NLP_COWORK.md) · graded labs in [`labs/part11/`](../../labs/part11/)

**You will:**
1. Build the word-list baseline and see where it fails.
2. Score 800 headlines with one Message Batches job and a cached rubric.
3. Turn scores into a point-in-time daily signal, using the time each score became available.
4. Measure what the signal is worth with a market-model event study, and what a look-ahead adds.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing calls the API: model outputs are recorded responses built as real `anthropic` Message objects, and the data is synthetic with known answers, so every check is exact and free.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p11lib.py is in notebooks/part11/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p11lib as p

p.use_course_style()

## 1. The baseline: a word list

Count positive and negative words: `(n_pos − n_neg) / (n_pos + n_neg)`, 0 when there are none (`p.POS_WORDS`, `p.NEG_WORDS`, lower-case words from `re.findall(r"[a-z]+", text.lower())`). Cheap, instant, transparent, and easy to fool.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import re

def lexicon_score(text):
    words = re.findall(r"[a-z]+", text.lower())
    n_pos, n_neg = ..., ...                                          # ✍️
    return (n_pos - n_neg) / (n_pos + n_neg) if n_pos + n_neg else 0.0

heads = ["ACME beats earnings estimates and raises outlook", "ACME shares jump as losses narrow sharply",
         "ACME fails to beat estimates despite record sales", "ACME to present at industry conference"]
mine = p.check("lexicon_score", [p.attempt(lexicon_score, h) for h in heads], [p.lexicon_score(h) for h in heads])
display(pd.Series(mine, index=heads, name="word-list score").to_frame())

"Losses narrow" is good news; "record sales" in a miss is bad news. The word list scores the first as neutral and the second as mildly positive.

## 2. One batch, one cached rubric

`p.batch_requests` builds one Message Batches request per **unique** headline (keyed by a hash of ticker, time and text, so duplicates are scored once). The long rubric sits in `system` with `cache_control`, identical in every request, so after the first request it is read from the cache; `output_config.format` forces a JSON answer. Batches cost half, and results come back in any order, keyed by `custom_id`.

In [ ]:
R, mkt, news = p.news_market()
rubric = "You score financial headlines for short-term price impact. " + "Rubric details… " * 200
schema = {"type": "object", "properties": {"sentiment": {"type": "number"}, "relevance": {"type": "number"}},
          "required": ["sentiment", "relevance"], "additionalProperties": False}
requests = p.batch_requests(pd.concat([news, news.head(50)]), rubric, schema)     # 50 repeated headlines
print(f"{len(news) + 50} headlines → {len(requests)} batch requests; first request:")
first = requests[0]
{**first, "params": {**first["params"], "system": [{**first["params"]["system"][0], "text": first["params"]["system"][0]["text"][:60] + "…"}]}}

## 3. When can a score be used?

The batch finishes about 3 hours after each headline, so a score is usable at `max(published_at, scored_at)`. To trade at the **open** (09:30) of each day, a score belongs to the first open **at or after** it became usable: `np.searchsorted(opens, times, side="left")` gives that position. Sum the sentiment per (day, ticker).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def daily_signal(scores, dates, time_col="usable_at"):
    opens = dates + pd.Timedelta(hours=9, minutes=30)
    pos = ...                                                        # ✍️ index of the first open at or after each time
    df = scores.assign(pos=pos)
    df = df[df["pos"] < len(dates)]
    sig = df.groupby(["pos", "ticker"])["sentiment"].sum().unstack(fill_value=0.0)
    return sig.reindex(range(len(dates)), fill_value=0.0).set_axis(dates)

R, mkt, news = p.news_market()
llm = p.recorded_llm_scores(news)                                   # scored by a batch job 3 hours after publication
llm["usable_at"] = np.maximum(llm["published_at"], llm["scored_at"])
cases = [(llm, R.index), (llm, R.index, "published_at")]
mine = [p.attempt(daily_signal, *c) for c in cases]
mine = p.check("daily_signal", mine, [p.daily_signal(*c) for c in cases])
signal_llm, signal_lookahead = mine
print(f"{len(news)} headlines → {int((signal_llm != 0).to_numpy().sum())} (stock, day) signals to trade at the open")

## 4. What is it worth?

First, accuracy on the headlines. Then a pooled **market-model event study** (`p.pooled_event_study`): after every (stock, day) with a strong signal, the cumulative abnormal return over the next 5 days. The LLM signal uses a threshold of 0.5; the word list, any non-zero score. The third row uses the **publication** time instead of the time the score existed: a look-ahead.

In [ ]:
truth, lex = news["true_sentiment"], news["headline"].map(p.lexicon_score)
acc = pd.DataFrame({"LLM": {"acts on": (llm["sentiment"].abs() > 0.5).mean(),
                            "sign right when it acts": (np.sign(llm["sentiment"]) == truth)[llm["sentiment"].abs() > 0.5].mean()},
                    "word list": {"acts on": (lex != 0).mean(), "sign right when it acts": (np.sign(lex) == truth)[lex != 0].mean()}})
display(acc.round(3))
signal_lex = p.daily_signal(news.assign(sentiment=lex, usable_at=news["published_at"]), R.index)
rows = {}
for name, sig, threshold in (("LLM, usable 3 h after the news", signal_llm, 0.5), ("LLM, publication time (look-ahead)", signal_lookahead, 0.5),
                             ("word list", signal_lex, 0.0)):
    for side in (True, False):
        path, t, n = p.pooled_event_study(R, mkt, sig, threshold, positive=side)
        rows[(name, "good news" if side else "bad news")] = {"5-day CAR": path.iloc[-1], "t-stat": t, "events": n}
pd.DataFrame(rows).T.round(4)

The LLM acts on more headlines and gets more of them right. After good news its signal earns more; after bad news the word list does at least as well, because negative words are rarely ironic and it acts at once, while the batch arrives three hours later and misses part of the first day. Pretending the scores existed at publication inflates the LLM's result: that gap is exactly what a live system could never earn. Measure before paying for the model, and always use the time a score *existed*.

## Wrap-up

* A word list is the baseline; the LLM must beat it on the thing you trade, not just on accuracy.
* Batch + cached rubric + dedup by hash: cheap scoring at scale.
* A score is usable when it exists, not when the news was published.
* Graded version: `labs/part11/week39_nlp_rag` (`lexicon_score`, `batch_requests`, `ScoreCache`, `daily_signal`, `event_study`).